# 🚗 Car Damage Assessment AI — YOLOv8 Training

**Phase 2: Training a real CV model on the CarDD dataset**

This notebook:
1. Installs YOLOv8 (ultralytics)
2. Downloads the CarDD dataset from Roboflow (YOLO format)
3. Trains YOLOv8n on 6 damage classes
4. Validates and shows metrics (mAP, precision, recall)
5. Tests on sample images
6. Exports `best.pt` for download

**6 damage classes:** dent, scratch, crack, glass_shatter, broken_lamp, flat_tire

---

⚡ **Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## Step 0 — Verify GPU

In [ ]:
# Verify GPU is available
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## Step 1 — Install dependencies

In [ ]:
# Install ultralytics (YOLOv8) and roboflow
!pip install -q ultralytics roboflow

from ultralytics import YOLO
import os
print("✅ Ultralytics installed successfully")

## Step 2 — Download CarDD dataset from Roboflow

The CarDD dataset contains ~4,000 car damage images with 9,000+ annotations across 6 damage categories.

We use the Roboflow-hosted version which is already in YOLOv8 format.

In [ ]:
from roboflow import Roboflow

# Download CarDD dataset in YOLOv8 format
# This is the public CarDD dataset hosted on Roboflow Universe
rf = Roboflow(api_key="h4f3OUamybWdCNxuLm1z")  # Free tier works fine
project = rf.workspace("cardd-diezp").project("detection-m16cd")
version = project.version(1)
dataset = version.download("yolov8")

print(f"\n✅ Dataset downloaded to: {dataset.location}")
print("\nDataset structure:")
!ls {dataset.location}
print("\nTrain images:")
!ls {dataset.location}/train/images | head -10
!echo "... Total: $(ls {dataset.location}/train/images | wc -l) images"

### ⚠️ Getting a Roboflow API key (free)

1. Go to [roboflow.com](https://roboflow.com) → Sign up (free)
2. Go to Settings → API Key
3. Copy the key and paste above

**Alternative: if you already have CarDD downloaded locally**, upload the zip to Colab and adjust the path in the training cell.

## Step 3 — Inspect dataset

In [ ]:
import yaml
from pathlib import Path

# Read and display data.yaml
data_yaml_path = f"{dataset.location}/data.yaml"
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("📋 Dataset configuration:")
print(f"   Classes: {data_config.get('names', 'N/A')}")
print(f"   Number of classes: {data_config.get('nc', 'N/A')}")
print(f"   Train path: {data_config.get('train', 'N/A')}")
print(f"   Val path: {data_config.get('val', 'N/A')}")
print(f"   Test path: {data_config.get('test', 'N/A')}")

# Count images
for split in ['train', 'valid', 'test']:
    img_dir = Path(dataset.location) / split / 'images'
    if img_dir.exists():
        count = len(list(img_dir.glob('*')))
        print(f"   {split}: {count} images")

In [ ]:
# Visualize a few training images with annotations
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

train_imgs = sorted((Path(dataset.location) / 'train' / 'images').glob('*'))[:6]
class_names = data_config.get('names', {})
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('CarDD Training Samples', fontsize=16, fontweight='bold')

for idx, (ax, img_path) in enumerate(zip(axes.flat, train_imgs)):
    img = Image.open(img_path)
    ax.imshow(img)

    # Load corresponding label
    label_path = str(img_path).replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                x_center, y_center, w, h = map(float, parts[1:5])

                # Convert YOLO format to pixel coordinates
                img_w, img_h = img.size
                x1 = (x_center - w/2) * img_w
                y1 = (y_center - h/2) * img_h
                box_w = w * img_w
                box_h = h * img_h

                color = colors[cls_id % len(colors)]
                rect = patches.Rectangle((x1, y1), box_w, box_h,
                    linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)

                cls_name = class_names[cls_id] if cls_id < len(class_names) else f'class_{cls_id}'
                ax.text(x1, y1 - 5, cls_name, color=color,
                    fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))

    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 4 — Train YOLOv8

Training config:
- **Model:** YOLOv8n (nano — fast, lightweight, good for demos and edge deployment)
- **Epochs:** 50 (good balance of quality and training time)
- **Image size:** 640px
- **Batch size:** 16 (fits T4 GPU memory)

Expected training time: **15–25 minutes** on T4 GPU.

In [ ]:
# Train YOLOv8 on CarDD dataset
model = YOLO('yolov8n.pt')  # Start from pretrained COCO weights

results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name='cardd_damage',
    project='runs/detect',
    patience=10,        # Early stopping if no improvement for 10 epochs
    save=True,
    save_period=10,     # Save checkpoint every 10 epochs
    plots=True,         # Generate training plots
    verbose=True,
)

print("\n✅ Training complete!")

## Step 5 — Evaluate model

In [ ]:
import glob
files = glob.glob('runs/**/best.pt', recursive=True)
print("Найденные модели:")
for f in files:
    print(f"  {f}")

In [ ]:
# Load the best model and run validation
best_model_path = 'runs/detect/runs/detect/cardd_damage/weights/best.pt'
model_best = YOLO(best_model_path)

# Validate on validation set
metrics = model_best.val(data=data_yaml_path)

print("\n" + "="*60)
print("📊 VALIDATION METRICS")
print("="*60)
print(f"  mAP@50:      {metrics.box.map50:.4f}")
print(f"  mAP@50-95:   {metrics.box.map:.4f}")
print(f"  Precision:   {metrics.box.mp:.4f}")
print(f"  Recall:      {metrics.box.mr:.4f}")
print("="*60)

# Per-class metrics
print("\n📋 Per-class mAP@50:")
class_names_list = list(data_config.get('names', {}).values()) if isinstance(data_config.get('names'), dict) else data_config.get('names', [])
for i, name in enumerate(class_names_list):
    if i < len(metrics.box.maps):
        print(f"  {name:20s} → mAP@50: {metrics.box.maps[i]:.4f}")

In [ ]:
# Display training curves
from IPython.display import Image as IPImage, display

results_dir = 'runs/detect/cardd_damage'

# Training results plot
results_img = f'{results_dir}/results.png'
if os.path.exists(results_img):
    print("📈 Training curves:")
    display(IPImage(filename=results_img, width=900))

# Confusion matrix
cm_img = f'{results_dir}/confusion_matrix.png'
if os.path.exists(cm_img):
    print("\n📊 Confusion matrix:")
    display(IPImage(filename=cm_img, width=600))

## Step 6 — Test on sample images

In [ ]:
# Run inference on validation images
val_imgs = sorted((Path(dataset.location) / 'valid' / 'images').glob('*'))[:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('YOLOv8 Predictions on Validation Set', fontsize=16, fontweight='bold')

for ax, img_path in zip(axes.flat, val_imgs):
    # Run prediction
    pred_results = model_best.predict(str(img_path), verbose=False, conf=0.25)

    # Get annotated image
    annotated = pred_results[0].plot()
    annotated_rgb = annotated[:, :, ::-1]  # BGR to RGB

    ax.imshow(annotated_rgb)

    # Count detections
    n_det = len(pred_results[0].boxes)
    ax.set_title(f'{img_path.name} ({n_det} damages)', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 7 — Export model

In [ ]:
import shutil

# Copy best weights to accessible location
src = 'runs/detect/runs/detect/cardd_damage/weights/best.pt'
dst = '/content/best.pt'
shutil.copy2(src, dst)

# Model info
file_size = os.path.getsize(dst) / (1024 * 1024)
print(f"✅ Model saved: {dst}")
print(f"   Size: {file_size:.1f} MB")
print(f"   mAP@50: {metrics.box.map50:.4f}")
print(f"   Classes: {class_names_list}")
print("\n📥 Download: click the file icon (left panel) → find best.pt → right-click → Download")
print("   Then put it in: Car-Damage-Assessment-AI/models/best.pt")

In [ ]:
import json
# Also save the notebook's training config for reproducibility
training_config = {
    'model': 'yolov8n',
    'dataset': 'CarDD (Roboflow)',
    'epochs': 50,
    'imgsz': 640,
    'batch': 16,
    'mAP50': round(float(metrics.box.map50), 4),
    'mAP50_95': round(float(metrics.box.map), 4),
    'precision': round(float(metrics.box.mp), 4),
    'recall': round(float(metrics.box.mr), 4),
    'classes': class_names_list,
    'weights_file': 'best.pt',
}

with open('/content/training_config.json', 'w') as f:
    json.dump(training_config, f, indent=2)

print("📋 Training config saved to /content/training_config.json")
print(json.dumps(training_config, indent=2))

In [ ]:
# Optional: download via Google Drive (for large files)
# Uncomment if direct download doesn't work

# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy2('/content/best.pt', '/content/drive/MyDrive/best.pt')
# shutil.copy2('/content/training_config.json', '/content/drive/MyDrive/training_config.json')
# print("✅ Files copied to Google Drive")

## ✅ Done!

### What to do next:

1. **Download** `best.pt` from the left panel (Files → right-click → Download)
2. **Download** `training_config.json` too
3. **Put them in your project:**
   ```
   Car-Damage-Assessment-AI/
   ├── models/
   │   ├── best.pt              ← trained weights
   │   └── training_config.json ← metrics & config
   ```
4. **Commit and push:**
   ```bash
   cd Car-Damage-Assessment-AI
   git add models/best.pt models/training_config.json
   git commit -m "feat: add YOLOv8 model trained on CarDD (mAP@50: X.XX)"
   git push origin main
   ```

### Metrics to put in README:
After training, update the README with actual numbers from the metrics above.